# RAG Demo - Retrieval Augmented Generation

Notebook para experimentar com RAG

In [ ]:
# Fix SQLite
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import os
sys.path.insert(0, os.path.abspath('..'))

## 1. Indexação de Documentos

In [ ]:
from src.document_processor import DocumentProcessor

processor = DocumentProcessor(
    collection_name="my_docs",
    persist_directory="../chroma_db"
)

# Liste seus arquivos em data/
!ls -lh ../data/

In [ ]:
# Indexe um documento
# processor.process_file('../data/seu_arquivo.pdf')

# Estatísticas
processor.get_collection_stats()

## 2. Retrieval - Busca Semântica

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def search(query, top_k=3):
    query_emb = embedding_model.encode([query])[0]
    results = collection.query(
        query_embeddings=[query_emb.tolist()],
        n_results=top_k
    )
    
    for i, doc in enumerate(results['documents'][0], 1):
        print(f"\n[{i}] {doc[:300]}...\n")
    
    return results

In [ ]:
# Teste de busca
results = search("sua pergunta aqui")

## 3. RAG Completo (requer LLM)

In [ ]:
from src.rag_chain import RAGChain

# Ollama
rag = RAGChain(
    collection_name="my_docs",
    persist_directory="../chroma_db",
    llm_provider="ollama"
)

# Ou OpenAI (se configurado)
# rag = RAGChain(llm_provider="openai")

In [ ]:
# Query RAG
result = rag.query("Sua pergunta aqui", return_context=True)

print("RESPOSTA:")
print(result['answer'])

print("\nCONTEXTO USADO:")
for doc in result['context_docs']:
    print(f"[{doc['rank']}] {doc['content'][:200]}...")

## 4. Experimentação

Teste diferentes parâmetros:

In [ ]:
# Diferentes tamanhos de chunk
processor_small = DocumentProcessor(chunk_size=500, chunk_overlap=50)
processor_large = DocumentProcessor(chunk_size=2000, chunk_overlap=400)

# Diferentes top_k
rag_k1 = RAGChain(top_k=1)
rag_k5 = RAGChain(top_k=5)